In [ ]:
# general
import sys 
import gc
from pathlib import Path
# virtual cell
import h5py
from gears import GEARS, PertData
import scanpy as sc
# data 
import numpy as np
# modelling 

# global settings
seed = 1733
np.random.seed(seed)
vcc_data_path = Path('vcc_data/VCC_filtered.h5ad')
gears_data_path = Path('gears_data_final_model')


## Data preperation

### Assumptions
We have right amount of perturbations (?) with at least > 40 samples per perturbation 
### Format
We create scanpy object and need to be:
- adata.obs
    - rename column: \
    'target_gene' -> 'condition'
    - change category: \
    'non-targeting' in 'condition' column -> 'ctrl'
    - change category: \
    'gene' in 'condition' column -> 'gene' + '+ctrl' - unperturbed
    - add column: \
    add column with cell type 'H1'
- adata.var
    - rename column: \
    'gene_id' -> 'gene_name'


### Loading data

IMPORTANT
- newer version hasn't some cmmand - we need to load old with out information of mask_var and base

In [ ]:
# repairing file - additional arguments:
with h5py.File(vcc_data_path, "r+") as f:
    # checking value
    print(list(f["uns/log1p"].keys()))  # powinno pokazać 'base'
    
    # getting read of value
    if "base" in f["uns/log1p"]:
        del f["uns/log1p/base"]

    if "mask_var" in f["uns/pca/params"]:
        print("Usuwam uns/pca/params/mask_var")
        del f["uns/pca/params/mask_var"]

gc.collect()

In [ ]:
# loading part of data
adata_big = sc.read_h5ad(vcc_data_path, backed='r')
adata_big

In [ ]:
# parameters
genes_amount =  1000
probes_per_gene = 10000

# mask
mask_ctrl = adata_big.obs['target_gene'] == 'non-targeting'
mask_rest = ~mask_ctrl
# taking random control cells
n_ctrl = 400
ctrl_idx = np.random.choice(
    np.where(mask_ctrl)[0],
    size=min(n_ctrl, mask_ctrl.sum()),
    replace=False
)

# applying mask
all_perts = adata_big.obs.loc[mask_rest, 'target_gene'].unique()
# taking random perturbations
chosen_perts = np.random.choice(all_perts, size=min(genes_amount, len(all_perts)), replace=False)

# taking random probes_per_gene cells from each perturbation
rest_idx_list = []
for pert in chosen_perts:
    idxs = np.where(adata_big.obs['target_gene'] == pert)[0]
    take_n = min(probes_per_gene, len(idxs))  # zabezpieczenie, jeśli jest mniej niż 40 komórek
    rest_idx_list.append(np.random.choice(idxs, size=take_n, replace=False))

# joining indices
final_idx = np.concatenate([ctrl_idx] + rest_idx_list)

In [ ]:
# loading data to memory
adata = adata_big.to_memory()

In [ ]:
# preprocessing
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
adata

In [ ]:
adata

In [ ]:
# creating right non-targeting -> ctrl
adata.obs.rename(columns={'target_gene': 'condition',}, inplace=True)
# converting data from categorical to string
adata.obs['condition'] = adata.obs['condition'].astype(str)

# changing format for gears model
if not 'ctrl' in adata.obs['condition'].values:
    print("Warning: 'ctrl' already present in conditions, renaming might cause issues.")
    # mask for 'non-targeting'
    mask_ctrl = adata.obs['condition'] == 'non-targeting'
    # changing values: 
    adata.obs.loc[mask_ctrl, 'condition'] = 'ctrl'
    adata.obs.loc[~mask_ctrl, 'condition'] = adata.obs.loc[~mask_ctrl, 'condition'] + '+ctrl'


# renaming columns
adata.obs['cell_type'] = 'H1'
adata.var.rename(columns={'gene_id': 'gene_name'}, inplace=True)

In [ ]:
adata

In [ ]:
# save and load data in gears format
pert_data = PertData(gears_data_path)
pert_data.new_data_process(dataset_name="vcc_data", adata=adata)
pert_data.load(data_path=str(gears_data_path/"vcc_data"))
# creating masks for training and validation
pert_data.prepare_split(split='simulation', train_gene_set_size=0.9)
# creating data loader for training
pert_data.get_dataloader(batch_size = 32, test_batch_size = 128)


## Model training 

### Library adjustment
in file:
```Bash
~/.pyenv/versions/scRNA_venv/lib/python3.10/site-packages/gears/gears.py
```
We need to change line 87 from:
```Python
np.mean(self.adata.X[(self.adata.obs.condition == 'ctrl')],
```
to:
```Python
np.mean(self.adata.X[(self.adata.obs.condition == 'ctrl').values],
```
Because we plugging data seriers into numpy array and they are not compatible 

In [ ]:
# laod prepared data
pert_data = PertData(gears_data_path)
pert_data.load(data_path=str(gears_data_path/"vcc_data"))
# creating masks for training and validation
pert_data.prepare_split(split='simulation', seed=1, )
# creating data loader for training
pert_data.get_dataloader(batch_size = 32, test_batch_size = 128)

In [ ]:
for i in range(7):
    # creating model instance
    gears_model = GEARS(pert_data, device = 'cuda', 
                            weight_bias_track = False, 
                            proj_name = 'vcc', 
                            exp_name = 'vcc')
    gears_model.model_initialize(hidden_size = 32 + 16*i)
    # training model
    gears_model.train(epochs = 20, lr = 1e-3)
    # saving model & loading model
    gears_model.save_model(f'test_model_{32+16*i}')

In [ ]:
# saving model & loading model
gears_model.save_model('test_model', )

In [ ]:

gears_model.load_pretrained('test_model')

In [ ]:
# making prediction, finding gene list 

# Fizyka - szybkie wykresy